# 01 — Tải & chuẩn bị dữ liệu trên Colab, lưu kết quả vào Drive

**Runtime: CPU** (Runtime → Change runtime type → CPU). Bước này chỉ tốn mạng và ổ đĩa, không
dùng GPU — bật T4 ở đây là phí hạn mức GPU. Để tab mở suốt lúc tải.

Vì sao ảnh US KHÔNG ghi thẳng vào Drive: ~60k ảnh nằm chung một thư mục `raw/images_us/`, mà
thao tác Drive trên Colab dễ timeout khi một thư mục có quá nhiều file (và ghi từng file nhỏ
rất chậm). Nên:

| Phần | `DATA_ROOT` khi chạy | Lưu lên Drive (`MyDrive/trademark_data/`) |
|---|---|---|
| US (train + test_us) | ổ máy ảo `/content/trademark_data` (nhanh, **mất khi hết phiên**) | `raw/meta/`, `raw/images_us.tar` (1 file), `processed/`, `lmdb/` |
| VN (test_vn) | thẳng trên Drive (~2k ảnh, cần nhiều phiên vì duyệt tay) | tất cả |

Notebook này tự clone repo + cài thư viện (máy ảo CPU là phiên riêng, không dùng chung với
`00_setup`). Bản cũ (ghi mọi thứ thẳng vào Drive): `01_download_data_old.ipynb`.

In [ ]:
# ---- tham số ----
REPO_URL = "https://github.com/nguyenlamvu123/major_proj_1.git"  # repo private: https://<token>@github.com/...
BRANCH = "16-fix-review"        # nhánh đã PUSH chứa code mới (đổi thành "main" sau khi merge)
US_META_ROWS = 60000            # dòng metadata US; sau khi lọc còn ~97% -> mục tiêu ≥ 50k mẫu sạch
VN_LIMIT = 2000                 # số nhãn hiệu VN (có chữ + có ảnh) cần crawl
VN_REVIEW_N = 300               # số nhãn VN duyệt tay

from google.colab import drive
drive.mount('/content/drive')
import os
LOCAL = "/content/trademark_data"
os.environ["DRIVE"] = "/content/drive/MyDrive/trademark_data"
os.environ["LOCAL"] = LOCAL
os.makedirs(os.environ["DRIVE"], exist_ok=True)

In [ ]:
!test -d /content/major_proj_1 || git clone -b {BRANCH} {REPO_URL} /content/major_proj_1
%cd /content/major_proj_1
!git log --oneline -1
!pip install -q -r requirements.txt && pip install -q -e .
!bash scripts/00_get_parseq.sh

## US — bước 1: metadata

Chạy lại sau khi phiên bị ngắt: cell này lấy lại bản sao trên Drive (nếu có) rồi `--resume`
— `--limit` tính cả dòng đã có nên chỉ tải phần còn thiếu.

In [ ]:
os.environ["DATA_ROOT"] = LOCAL          # phần US chạy trên ổ máy ảo
!mkdir -p $LOCAL/raw $DRIVE/raw
!test -d $DRIVE/raw/meta && cp -rn $DRIVE/raw/meta $LOCAL/raw/ || true
!test -f $DRIVE/raw/images_us.tar && tar -xf $DRIVE/raw/images_us.tar -C $LOCAL/raw || true
!bash scripts/01_get_meta.sh --limit {US_META_ROWS} --resume
!cp -r $LOCAL/raw/meta $DRIVE/raw/

## US — bước 2: ảnh (lâu nhất)

Ảnh đã tải (có trong `index.csv` + file tồn tại) được bỏ qua, nên chạy lại là tải tiếp. Xong là
**sao lưu ngay** thành MỘT file tar trên Drive — mất phiên trước bước này thì phải tải lại ảnh.

In [ ]:
os.environ["DATA_ROOT"] = LOCAL
!bash scripts/02_get_images.sh --workers 16
!python -m trademark_ocr.fetch_images --verify
!tar -cf $DRIVE/raw/images_us.tar -C $LOCAL/raw images_us && cp -r $LOCAL/raw/meta $DRIVE/raw/
!ls -lh $DRIVE/raw/images_us.tar

## US — bước 3: lọc → chia tập → LMDB → chép lên Drive

In [ ]:
os.environ["DATA_ROOT"] = LOCAL
!python -m trademark_ocr.clean
!python -m trademark_ocr.split --seed 1111
!for s in train val test_us; do python -m trademark_ocr.build_lmdb --subset $s; done
# không --delete: giữ phần VN (processed/*vn*, lmdb/test/test_vn) đã có trên Drive
!rsync -a $LOCAL/processed $LOCAL/lmdb $DRIVE/

In [ ]:
# Kiểm tra bản trên Drive: mỗi dòng phải là một con số (không phải None) = số dòng split tương ứng
!for d in train/trademark val test/test_us; do python -c "import lmdb,sys;print(sys.argv[1], lmdb.open(sys.argv[1],readonly=True,lock=False).begin().get(b'num-samples'))" $DRIVE/lmdb/$d; done
!python -c "import json,os;r=json.load(open(os.environ['DRIVE']+'/processed/split_report.json'));print(r['counts'])"

## VN — crawl, duyệt tay, LMDB (làm thẳng trên Drive)

1. Chạy cell dưới: crawl `VN_LIMIT` nhãn và sinh bảng duyệt `processed/review_vn.csv` +
   trang xem `processed/review_vn.html`.
2. Từ Drive, tải hai thư mục `trademark_data/processed` và `trademark_data/raw/images_vn` về
   máy, **giữ nguyên cấu trúc** (HTML trỏ ảnh bằng đường dẫn `../raw/images_vn/...`). Mở
   `review_vn.html` để xem ảnh cạnh nhãn.
3. Sửa cột `visible_text` trong `review_vn.csv` cho đúng chữ nhìn thấy trong ảnh (giữ nguyên
   nếu đúng, để **trống** nếu ảnh không dùng được), rồi upload đè vào `processed/` trên Drive.
4. Chạy cell "áp kết quả duyệt" (phiên mới cũng được — chỉ cần chạy 2 cell đầu notebook).

`make` không ghi đè `review_vn.csv` đã có (tránh mất công duyệt); muốn lấy mẫu lại thì thêm
`--overwrite`.

In [ ]:
os.environ["DATA_ROOT"] = os.environ["DRIVE"]
!bash scripts/03_get_vn.sh --limit {VN_LIMIT}
!python -c "import json,os;r=json.load(open(os.environ['DRIVE']+'/raw/vn_report.json'));print(r['n'],'nhãn |',r['n_registered'],'đã cấp |',r['skipped'])"
!python -m trademark_ocr.vn_review make --n {VN_REVIEW_N}

In [ ]:
# ---- áp kết quả duyệt (sau khi đã upload review_vn.csv đã sửa) ----
os.environ["DATA_ROOT"] = os.environ["DRIVE"]
!python -m trademark_ocr.vn_review apply
!python -m trademark_ocr.build_lmdb --subset test_vn --split-csv $DRIVE/processed/split_test_vn_verified.csv
!python -c "import lmdb,os;print('test_vn', lmdb.open(os.environ['DRIVE']+'/lmdb/test/test_vn',readonly=True,lock=False).begin().get(b'num-samples'))"

## Xong — trên Drive (`MyDrive/trademark_data/`)

```
raw/meta/us_marks.csv          raw/images_us.tar        raw/images_vn/  raw/vn_marks.csv
processed/  (clean, split_*, review_vn*, split_test_vn_verified.csv, *_report.json)
lmdb/train/trademark  lmdb/val  lmdb/test/test_us  lmdb/test/test_vn
```

Ghi `label_noise_rate` (trong `processed/review_vn_report.json`) vào báo cáo đồ án.

Bước tiếp: đổi runtime sang **T4 GPU** → `00_setup.ipynb` → `02_train.ipynb` (đặt
`DATA_ROOT` = Drive và tự chép `lmdb/` ra ổ máy ảo trước khi train).